<a href="https://colab.research.google.com/github/vanderbilt-data-science/MNPSCollaborative/blob/Restart-From-Hackathon_v2.0/mnps_post_getting_started%20v4.8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **MNPS Job Equity Post Mini-Hackathon 4.8**
> A notebook that builds off of the work done in the hackathon.  
> DSI DSSG + MNPS Hackathon  
> September 10, 2025  
> Drafted by Wayne Birch - [contact him](wayne.birch@mnps.org) for questions, code update needs, or other questions about the notebook!

This notebook is a restart point based on the work done in the mini Hackathon with Metro Nashville Public Schools (MNPS) and the VU Data Science Institute (VU DSI).

 **Competition Details from the Hackathon with some updates follow:**

You aren't constrained to what is in this notebook, and please feel free to use your creativity to deliver the best solution
# **1** | Competition Parameters
* **Outcome and evaluation:** Participants will be evaluated on the performance of their provided solution on the holdout set. Importantly, judges must be able to easily run the submitted code on the new dataset.
* **Objective:** The overall objective is to create a system which best automatically, reproducibly, and reliably categorizes jobs according to the parameters set forth by MNPS. A few suggestions are provided on parameters that you can vary if you're thinking about achievable changes in 2.5 hours



## **2** | Environment Setup
Again, you're completely free to just download this notebook, create a local virtual environment and get to coding in your favorite IDE. We provide this code just as a rapid method to get started, and focus our efforts on implementation through Google Colab.

### **2a** | API Key Setup
#### **2a.1** | Access
The DSI has provided you an API key which can access **some** of the OpenAI models. These include:
* All versions of gpt-4o
* All versions of gpt-4.1
* All versions of o3-mini

Vector store upload, web search, code interpreter, and other functionality outside of the Chat Completions and Messages API is **not** supported. If you really want to use these things, you will have to make a good and cost-supported argument. If you don't feel like arguing, you can also utilize your own OpenAI API key.

#### **2a.2** | API Keys in Google Colab
To use your API key, click on the key icon (looks sort of like 🔑) in the left sidebar.  Under **Name**, add `OPENAI_API_KEY`. Under **Value**, paste your API key. Your API key is a jumble of numbers and letters, maybe even other symbols. Click the slider checkbox to enable **Notebook access** (so your notebook will grab these values without asking you).  

### **2b** | Runtime setup
We're going to install some packages in your environment so that you have access to the code functionality. If you need more packages, install more packages. Install **only** packages you trust.


> # **Version 4.2 Change**
> Pinned model snapshots for version control  
> Added API Key Secret of model versioning


In [ ]:
#Cell 3
!pip install -U openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 945.2/945.2 kB 16.0 MB/s eta 0:00:00
  Attempting uninstall: openai
    Found existing installation: openai 1.106.1
    Uninstalling openai-1.106.1:
      Successfully uninstalled openai-1.106.1


In [ ]:
#Cell 3.5
# ===== Environment Setup (single source of truth) =====
import os
from typing import List
import pandas as pd
from pydantic import BaseModel, Field
from google.colab import userdata

# 1) API key from Colab's 🔑 panel
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

# 2) Read the model selector from Colab's 🔑 panel (can be alias or snapshot)
RAW_MODEL = userdata.get("OPENAI_MODEL")  # e.g., gpt-4o, gpt-4o-2024-11-20, gpt4.1, o3 mini

def normalize_model_id(s: str | None) -> str | None:
    if not s:
        return None
    s = s.strip().lower().replace("_", "-").replace(" ", "-")
    fixes = {
        "gpt4o": "gpt-4o",
        "gpt-4o": "gpt-4o",
        "gpt4.1": "gpt-4.1",
        "gpt-41": "gpt-4.1",
        "o3mini": "o3-mini",
        "o3-mini": "o3-mini",
    }
    return fixes.get(s, s)

alias_or_snapshot = normalize_model_id(RAW_MODEL)

# 3) Map aliases → pinned snapshots you prefer (edit to taste)
SNAPSHOTS = {
    # GPT-4o snapshots (stable; good for Structured Outputs)
    "gpt-4o":  "gpt-4o-2024-11-20",
    # GPT-4.1 family snapshot (long context)
    "gpt-4.1": "gpt-4.1-2025-04-14",
    # Keep o3-mini as an alias (no public dated snapshot ID); good for reasoning
    "o3-mini": "o3-mini",
}

# 4) Final MODEL_ID selection rule:
#    - If user entered an alias, pin it via SNAPSHOTS
#    - If user entered a snapshot, pass it through
#    - Else fallback to a safe default snapshot
MODEL_ID = SNAPSHOTS.get(alias_or_snapshot or "", None) or (alias_or_snapshot) or "gpt-4o-2024-11-20"

print("🔧 OPENAI_MODEL (raw):", RAW_MODEL)
print("✅ Using MODEL_ID:", MODEL_ID)


🔧 OPENAI_MODEL (raw): GPT-4o
✅ Using MODEL_ID: gpt-4o-2024-11-20


> # **Version 4.2.1 Change**
> Ignore "Job Description Name" in input file  
> Print string for records going to API


In [ ]:
# ===== Cell 4 — Unique run folder + get inputs (3 files) + robust CSV read =====
import os, json, shutil, datetime as dt, zipfile
from pathlib import Path
import pandas as pd
from google.colab import drive
from openai import OpenAI

# ---------- 0) Mount Drive ----------
drive.mount('/content/drive')

# ---------- 1) Fixed output location ----------
RUN_ROOT = Path("/content/drive/My Drive/Colab Notebooks/Run Results")
timestamp = dt.datetime.utcnow().strftime("%Y%m%d_%H%M%S")
RUN_DIR = RUN_ROOT / f"RUN_{timestamp}"
INPUTS_DIR = RUN_DIR / "inputs"
OUTPUTS_DIR = RUN_DIR / "outputs"
for p in (RUN_DIR, INPUTS_DIR, OUTPUTS_DIR):
    p.mkdir(parents=True, exist_ok=True)

print("🗂️ Run folder:", RUN_DIR)

# ---------- 2) Where to find your three inputs by default ----------
# If you want to upload instead of copying from Drive, set ALLOW_UPLOAD = True.
DATA_INPUTS_DIR = Path("/content/drive/My Drive/Colab Notebooks/Data Inputs")
UNZIPPED_INPUTS_DIR = Path("/content/") # Added check for unzipped files
ALLOW_UPLOAD = False  # set True to upload the 3 files from your computer (CSV, CSV, ZIP)

REQUIRED = {
    "Ground Truth Masterfile.csv": DATA_INPUTS_DIR / "Ground Truth Masterfile.csv",
    "New Sample_08.07.2025.csv":  DATA_INPUTS_DIR / "New Sample_08.07.2025.csv",
    "MNPS_Prompt_Resources.zip":  DATA_INPUTS_DIR / "MNPS_Prompt_Resources.zip",
}

# (A) Optionally upload files instead of copying from Drive
if ALLOW_UPLOAD:
    from google.colab import files as colab_files
    print("🔼 Upload the three files when prompted:")
    uploaded = colab_files.upload()  # opens a browser picker
    for name in REQUIRED.keys():
        if name in uploaded:
            dst = INPUTS_DIR / name
            with open(dst, "wb") as f:
                f.write(uploaded[name])
            REQUIRED[name] = dst  # point to the just-uploaded copy

# (B) Copy from Drive or use unzipped files if not already present in /inputs
missing = []
for name, src in REQUIRED.items():
    dst = INPUTS_DIR / name
    if dst.exists():
        continue
    if src.exists():
        shutil.copy2(src, dst)
        print(f"📄 Copied: {src}  →  {dst}")
    elif (UNZIPPED_INPUTS_DIR / name).exists(): # Check in unzipped directory
        shutil.copy2(UNZIPPED_INPUTS_DIR / name, dst)
        print(f"📄 Copied from unzipped: {UNZIPPED_INPUTS_DIR / name}  →  {dst}")
    else:
        missing.append(name)

if missing:
    raise FileNotFoundError(
        "These input files were not found. Place them in "
        f"{DATA_INPUTS_DIR}, enable ALLOW_UPLOAD=True, or ensure they are unzipped to /content/: \n - " + "\n - ".join(missing)
    )

# ---------- 3) Unpack the resources zip into inputs/resources (optional but helpful) ----------
resources_zip = INPUTS_DIR / "MNPS_Prompt_Resources.zip"
RESOURCES_DIR = INPUTS_DIR / "resources"
if resources_zip.exists():
    RESOURCES_DIR.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(resources_zip, "r") as zf:
        zf.extractall(RESOURCES_DIR)
    print("🧰 Unpacked resources to:", RESOURCES_DIR)

# ---------- 4) Robust CSV reader (handles cp1252/latin1) ----------
def read_csv_smart(path: Path, **kwargs) -> pd.DataFrame:
    trials = [
        dict(encoding="utf-8"),
        dict(encoding="utf-8-sig"),
        dict(encoding="cp1252"),
        dict(encoding="latin1"),
    ]
    for t in trials:
        try:
            df = pd.read_csv(path, **{**t, **kwargs})
            print(f"✅ Read {path.name} with encoding={t['encoding']}")
            return df
        except UnicodeDecodeError:
            continue
    # last resort
    df = pd.read_csv(path, encoding="latin1", on_bad_lines="skip", **kwargs)
    print(f"⚠️ Read {path.name} with encoding=latin1 (on_bad_lines='skip')")
    return df

# ---------- 5) Load sample CSV and build the single-row job_desc_text ----------
sample_csv = INPUTS_DIR / "New Sample_08.07.2025.csv"
df = read_csv_smart(sample_csv)

required_cols = [
    "Job Description Name",
    "Position Summary",
    "Education",
    "Work Experience",
    "Essential Functions",
    "Licenses and Certifications",
    "Knowledge, Skills and Abilities",
]
missing_cols = [c for c in required_cols if c not in df.columns]
if missing_cols:
    raise ValueError(f"Missing required columns in {sample_csv.name}: {missing_cols}")

ROW_IDX = 0
r = df.iloc[ROW_IDX]

# include Job Description Name so Cell 16 can pick it up and then ignore/anonymize it
job_desc_text = f"""Job Description:
Job Description Name: {r['Job Description Name']}

Position Summary: {r['Position Summary']}
Education: {r['Education']}
Work Experience: {r['Work Experience']}
Licenses and Certifications: {r['Licenses and Certifications']}
Essential Functions: {r['Essential Functions']}
Knowledge, Skills and Abilities: {r['Knowledge, Skills and Abilities']}
"""
print("🧪 Prepared job_desc_text from row", ROW_IDX)

# ---------- 6) (Optional) Upload to OpenAI — DISABLED for text-only pipeline ----------
ALLOW_OPENAI_UPLOAD = False
file_ids = []

if ALLOW_OPENAI_UPLOAD:
    client = OpenAI()
    to_upload = [
        INPUTS_DIR / "Ground Truth Masterfile.csv",
        INPUTS_DIR / "New Sample_08.07.2025.csv",
    ]
    uploaded = []
    for p in to_upload:
        with open(p, "rb") as f:
            up = client.files.create(file=f, purpose="assistants")
        uploaded.append(up)
    file_ids = [u.id for u in uploaded]
    print("⬆️ Uploaded file_ids:", file_ids)
else:
    print("⏭️ Skipping file upload to OpenAI (text-only pipeline).")

# ---------- 7) Write a small manifest so you can audit each run ----------
manifest = {
    "run_folder": str(RUN_DIR),
    "created_utc": timestamp,
    "inputs": [str(INPUTS_DIR / "Ground Truth Masterfile.csv"),
               str(INPUTS_DIR / "New Sample_08.07.2025.csv")],
    "resources_dir": str(RESOURCES_DIR) if RESOURCES_DIR.exists() else None,
    "uploaded_file_ids": file_ids,
    "text_only_pipeline": True,
}
(RUN_DIR / "RUN_METADATA.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")

print("\n📁 Current run tree (first few entries):")
for i, p in enumerate(sorted(RUN_DIR.rglob("*"))):
    print(" -", p.relative_to(RUN_DIR))
    if i > 25:
        print(" … (truncated)")
        break

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
🗂️ Run folder: /content/drive/My Drive/Colab Notebooks/Run Results/RUN_20250910_191314
📄 Copied: /content/drive/My Drive/Colab Notebooks/Data Inputs/Ground Truth Masterfile.csv  →  /content/drive/My Drive/Colab Notebooks/Run Results/RUN_20250910_191314/inputs/Ground Truth Masterfile.csv
📄 Copied: /content/drive/My Drive/Colab Notebooks/Data Inputs/New Sample_08.07.2025.csv  →  /content/drive/My Drive/Colab Notebooks/Run Results/RUN_20250910_191314/inputs/New Sample_08.07.2025.csv
📄 Copied: /content/drive/My Drive/Colab Notebooks/Data Inputs/MNPS_Prompt_Resources.zip  →  /content/drive/My Drive/Colab Notebooks/Run Results/RUN_20250910_191314/inputs/MNPS_Prompt_Resources.zip
🧰 Unpacked resources to: /content/drive/My Drive/Colab Notebooks/Run Results/RUN_20250910_191314/inputs/resources
✅ Read New Sample_08.07.2025.csv with encoding=cp1252
🧪 Prepared job_desc_t

/tmp/ipython-input-3528295825.py:13: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp = dt.datetime.utcnow().strftime("%Y%m%d_%H%M%S")


In [ ]:
# Cell 6
from openai import OpenAI
client = OpenAI()

visible = {m.id for m in client.models.list().data}
if MODEL_ID not in visible:
    print(f"⚠️ {MODEL_ID} is not visible to your key. "
          "Use an alias you do see (e.g., gpt-4o) or confirm access in your org.")
else:
    print(f"👍 {MODEL_ID} is available.")


👍 gpt-4o-2024-11-20 is available.


## **3** | The Data

The current prompt is a two-step prompt that is successful through the ChatGPT interface. It requires two types of data:
* The data to be classified
* Supporting resources

We need to read all of this in. Let's grab it and use it. The first thing you'll do is just straight up download a zip file of all of this information.

You can download all of the reference files from the link provided, then upload in the sidebar. You'll then unzip the directory using the code below.

Click on the folder icon in the left sidebar (kinda looks like this 🗂️) and you'll see all the files there. We'll read them in.

# **Change for Version 4.2**
> Gets input files from Google Drive folder and unzips for use in /content/  

In [ ]:
# Cell 8
from google.colab import drive
drive.mount('/content/drive')
base_target_folder = '/content/drive/My Drive/Colab Notebooks/Data Inputs'
!unzip "{base_target_folder}/MNPS_Prompt_Resources.zip" -d /content/

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Archive:  /content/drive/My Drive/Colab Notebooks/Data Inputs/MNPS_Prompt_Resources.zip
  inflating: /content/Korn_Ferry Lominger 38 Competencies.csv  
  inflating: /content/Competency Extended Descriptions.csv  
  inflating: /content/MNPS KSACs.csv  
  inflating: /content/MNPS Roles.csv  


## **4** | The Prompts

What we have here is a direct prompt to get the response that we're looking for. We'll make this happen directly using the OpenAI Chat Completions API. Note that you can use other APIs as you like.

In [ ]:
#Cell 12
zero_shot_prompt = \
""" Objective: Evaluate and group jobs from the "New Sample_08.07.2025.csv" file based on similarities in job functions, not job titles.

Process:

- Compare all jobs against each other using the attributes listed in the file: Education, Work Experience, Licenses/Certifications, Essential Functions, Knowledge, Skills, Abilities, and Position Summary.
- Compare each job with reference sources using the same attributes. I have attached the reference sources for you.
- Group jobs based on similarities into:
  - Major role groupings (e.g., Specialist, Analyst, Manager)
  - Minor sub-groupings (e.g., I, II, III, IV) - not to exceed level IV
- Use the MNPS Roles and MNPS KSACs documents to help you determine major role groupings.
- Use the remaining documents to help you clarify subtle differences in role groupings and sub-groupings.
- Use a more qualitative, holistic assessment focused on functional alignment with KSACs rather than a quantitative scoring approach with defined complexity metrics

Output Format:

- Create a table with the following columns:
  - Original Job Title
  - New Job Title
  - Major Role Group
  - Minor Sub-Group
  - Justification for Grouping

- Provide an accompanying narrative explaining the rationale behind the groupings and any notable patterns or insights discovered during the analysis.

Job Title Convention:

- Follow the format: "[Function] [Role] [Level]" (e.g., "Collections Specialist II", "Accounts Payable Specialist III")

Additional Guidelines:

- Ensure all sources used are cited properly.
- Focus on the nature of the work performed rather than just the job titles.
- Consider the complexity of tasks, level of responsibility, and required competencies when determining groupings.
- Provide clear explanations for why each job was classified as it was, referencing specific job attributes and external benchmarks.

"""

Instead of asking for a table output, we will use **structured outputs**. Though this is a common approach for the outputs of LLMs/AI systems, you can learn more about this on [OpenAI's structured output documentation](https://platform.openai.com/docs/guides/structured-outputs?api-mode=responses). Note that you can find this information on almost all LLM/AI platform or package providers.

In [ ]:
#Cell 14
from pydantic import BaseModel, Field

class JobClassification(BaseModel):
    """Represents the classification of a job based on its functions."""
    job_title_original: str = Field(..., description="The original job title as provided in the input data using the job title convention specified.")
    new_job_title: str = Field(..., description="The proposed new job title based on the classification using the job title convention specified.")
    major_role_group: str = Field(..., description="The major grouping of the job based on its functional role (e.g., Specialist, Analyst, Manager).")
    minor_sub_group: str = Field(..., description="The minor sub-grouping within the major role group (e.g., Specialist I, II, III, IV).")
    grouping_justification: str = Field(..., description="The justification for placing the job in the specific major and minor groups, referencing job attributes and relevant documents.")

In [ ]:
#Cell 15
from typing import List

class JobClassificationTable(BaseModel):
  """The table classification and overall commentary on the groupings provided by the AI system."""
  job_classification_table: List[JobClassification] = Field(..., description="The table of job classifications.")
  narrative_rationale: str = Field(..., description="The narrative commentary on the groupings provided by the AI system.")

Create classifications using OpenAI. Of note here is:
* The **developer** prompt - this is the "system prompt" or "custom instructions" for the model. This determines the overall behavior of the model.
* The **user** prompt - this is what we send to the model like when we're chatting with ChatGPT.

# **Changes for Verions 4.2**
> Responses API (TEXT-ONLY, no attachments), saves to OUTPUTS_DIR  
> Composes text-only input (no attachments)  
> Newer SDK: server-enforced  
> Structured Outputs via parse; uses jsons  
> Save outputs to OUTPUTS_DIR


# **Changes for Verions 4.2**
>  Pre-flight: are all prerequisites loaded for batch


In [ ]:
# ===== Cell 16 — Single row (role-constrained, anonymize PS/EF, Lead minor level, save payload) =====
from openai import OpenAI
from pathlib import Path
import pandas as pd, json, inspect, re

client = OpenAI()

# ---- Requirements from earlier cells ----
assert 'MODEL_ID' in globals(), "Run Environment Setup first (MODEL_ID)."
assert 'OUTPUTS_DIR' in globals() and 'INPUTS_DIR' in globals(), "Run Cell 4 first."
assert 'zero_shot_prompt' in globals() and 'job_desc_text' in globals(), "Make sure Cell 4 built job_desc_text."
print("🤖 Using model:", MODEL_ID)

# --- helpers (JSON fence cleanup) ---
_FENCE_RE = re.compile(r"^\s*```(?:json)?\s*(.*?)\s*```\s*$", re.I | re.S)
_BRACE_RE = re.compile(r"\{.*\}", re.S)
def coerce_to_json_str(raw: str) -> str:
    if not isinstance(raw, str): return ""
    s = raw.strip()
    m = _FENCE_RE.match(s)
    if m: return m.group(1).strip()
    b = _BRACE_RE.search(s)
    return b.group(0).strip() if b else s

# ---- allowed majors from MNPS Roles.csv (must exist in INPUTS_DIR created by Cell 4) ----
def read_roles_allowed() -> list[str]:
    # Check in both INPUTS_DIR and the unzipped /content/
    path_in_inputs = Path(INPUTS_DIR) / "MNPS Roles.csv"
    path_in_content = Path("/content/") / "MNPS Roles.csv" # Corrected path
    path = None
    if path_in_inputs.exists():
        path = path_in_inputs
    elif path_in_content.exists():
        path = path_in_content

    if not path:
        raise FileNotFoundError(
            f"MNPS Roles.csv not found in {path_in_inputs} or {path_in_content}. Re-run Cell 4 (it copies your inputs into this run) or Cell 8 (it unzips resources)."
        )
    df = pd.read_csv(path, encoding="cp1252")
    cand = [c for c in df.columns if "major" in c.lower() and "role" in c.lower()] \
        or [c for c in df.columns if "role" in c.lower()]
    col = cand[0]
    roles = sorted({str(x).strip() for x in df[col].dropna().unique() if str(x).strip()})
    return roles

ALLOWED_MAJORS = read_roles_allowed()
ALLOWED_MAJORS_SET = set(ALLOWED_MAJORS)
MINOR_ALLOWED = {"", "I", "II", "III", "LEAD"}

# ---- title & anonymization (ignore title for eval; replace PS/EF title mentions with "this role") ----
title_match = re.search(r"^Job Description Name:\s*(.+)$", job_desc_text, flags=re.M)
original_title = title_match.group(1).strip() if title_match else ""

def anonymize_title_in_text(title: str, text: str) -> str:
    if not title: return text
    toks = [t for t in title.strip().split() if t]
    if not toks: return text
    pattern = r"(?i)\b" + r"(?:[\s\-]+)".join(map(re.escape, toks)) + r"(?:'s)?\b"
    return re.sub(pattern, "this role", text)

lines_out, ps_line, ef_line = [], "", ""
for line in job_desc_text.splitlines():
    if line.startswith("Job Description Name:"):
        continue
    if line.startswith("Position Summary:"):
        val = line.split(":", 1)[1].lstrip()
        ps_line = "Position Summary: " + anonymize_title_in_text(original_title, val)
        line = ps_line
    elif line.startswith("Essential Functions:"):
        val = line.split(":", 1)[1].lstrip()
        ef_line = "Essential Functions: " + anonymize_title_in_text(original_title, val)
        line = ef_line
    lines_out.append(line)
jd_body_sanitized = "\n".join(lines_out).strip()
students_hint = ("students" in (ps_line.lower() + " " + ef_line.lower()))

# ---- constraints in prompt ----
constraints = (
    "Hard constraints:\n"
    f"- Major Role Group MUST be one of: {', '.join(ALLOWED_MAJORS)}\n"
    "- Use MNPS Roles and MNPS KSACs as the primary basis for Major Role Group.\n"
    "- Use Competency Extended Descriptions and Korn/Ferry 38 Competencies only to clarify/support the justification.\n"
    "- Use Ground Truth Masterfile as a tie-breaker when multiple roles are plausible by KSACs.\n"
    "- Minor Sub-Group must be one of: \"\", I, II, III, Lead (no higher than Lead). Use Lead only when the record clearly indicates a level IV scope.\n"
    "- For Teacher vs Instructor: if KSAC alignment is equal and the text references students, prefer Teacher.\n"
    "- For advisory-type language (consulting/advising/coaching/facilitating), choose among Advisor / Consultant / Coach / Facilitator based on KSAC alignment and determinant factors; justify with KSAC evidence.\n"
)

if students_hint:
    constraints += "Note: The text references students; prefer Teacher vs Instructor when KSACs are equivalent.\n"

full_text = (
    zero_shot_prompt.strip()
    + "\n\n" + constraints
    + "\nClassify the following job description (title ignored for evaluation):\n\n"
    + jd_body_sanitized
)

# ---- save the exact payload ----
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)
payload_single = Path(OUTPUTS_DIR) / "API_Payload_SINGLE.txt"
payload_single.write_text(full_text, encoding="utf-8")

# ---- capability detection ----
def _has_param(obj, name: str) -> bool:
    try: return name in inspect.signature(obj).parameters
    except Exception: return False

supports_parse_schema  = _has_param(client.responses.parse,  "response_format")
supports_create_schema = _has_param(client.responses.create, "response_format")

# ---- call model ----
parsed, raw_text = None, ""
try:
    if supports_parse_schema:
        resp = client.responses.parse(
            model=MODEL_ID,
            input=[{"role":"user","content":[{"type":"input_text","text":full_text}]}],
            temperature=0.2, max_output_tokens=1400,
            response_format=JobClassificationTable,
        )
        parsed, raw_text = resp.output_parsed, (resp.output_text or "")
    elif supports_create_schema:
        schema = JobClassificationTable.model_json_schema()
        resp = client.responses.create(
            model=MODEL_ID,
            input=[{"role":"user","content":[{"type":"input_text","text":full_text}]}],
            temperature=0.2, max_output_tokens=1400,
            response_format={"type":"json_schema","json_schema":{"name":"JobClassificationTable","schema":schema,"strict":True}},
        )
        raw_text = coerce_to_json_str(getattr(resp,"output_text",None) or "")
        parsed = JobClassificationTable.model_validate_json(raw_text) if raw_text else None
    else:
        schema_json = json.dumps(JobClassificationTable.model_json_schema(), indent=2)
        strict = (
            "You MUST return ONLY valid JSON matching the JSON Schema. No prose, no markdown.\n"
            f"JSON Schema:\n{schema_json}\n\nTask:\n{full_text}"
        )
        resp = client.responses.create(
            model=MODEL_ID,
            input=[{"role":"user","content":[{"type":"input_text","text":strict}]}],
            temperature=0.2, max_output_tokens=1400,
        )
        raw_text = coerce_to_json_str(getattr(resp,"output_text",None) or "")
        parsed = JobClassificationTable.model_validate_json(raw_text) if raw_text else None
except Exception as e:
    print("❗ Responses API error:", e)
    raise

# ---- post-normalize (Lead + teacher/instructor; NO forced Consultant->Advisor) ----
def norm_minor(s: str) -> str:
    if not s: return ""
    x = str(s).strip().upper().replace("LEVEL ","")
    map_ = {"1":"I","I":"I","2":"II","II":"II","3":"III","III":"III","4":"LEAD","IV":"LEAD","LEAD":"LEAD"}
    return map_.get(x, x if x in MINOR_ALLOWED else "")

def choose_teacher_instructor(cur: str) -> str:
    cur = (cur or "").strip()
    if students_hint and ("Teacher" in ALLOWED_MAJORS_SET):
        return "Teacher"
    return cur

def norm_major(s: str) -> str:
    if not s: return s
    t, low = s.strip(), s.strip().lower()
    if low in {"teacher","instructor"}:
        t = choose_teacher_instructor(t)
    if t in ALLOWED_MAJORS_SET: return t
    for a in ALLOWED_MAJORS:
        if a.lower() == low: return a
    tok = set(re.findall(r"\w+", low))
    best,score = None,-1
    for a in ALLOWED_MAJORS:
        sc = len(tok & set(re.findall(r"\w+", a.lower())))
        if sc>score: best,score=a,sc
    return best or t

raw_path = Path(OUTPUTS_DIR) / "Raw_Response_SINGLE.json"
raw_path.write_text(raw_text or "", encoding="utf-8")
print("✅ Saved:", raw_path)

if parsed is not None:
    rows = [row.model_dump() for row in parsed.job_classification_table]
    if rows:
        rows[0]["job_title_original"] = original_title or rows[0].get("job_title_original","")
        rows[0]["minor_sub_group"]    = norm_minor(rows[0].get("minor_sub_group",""))
        rows[0]["major_role_group"]   = norm_major(rows[0].get("major_role_group",""), students_hint)
    out_csv = Path(OUTPUTS_DIR) / "Job_Classifications_SINGLE.csv"
    pd.DataFrame(rows).to_csv(out_csv, index=False, encoding="utf-8")
    out_txt = Path(OUTPUTS_DIR) / "Narrative_SINGLE.txt"
    out_txt.write_text(parsed.narrative_rationale, encoding="utf-8")
    print("✅ Saved:", out_csv)
    print("✅ Saved:", out_txt)
else:
    print("⚠️ No parsed object returned (see Raw_Response_SINGLE.json).")

print("\n=== PAYLOAD SENT (first 800 chars) ===")
print(payload_single.read_text(encoding="utf-8")[:800], "...\n")

🤖 Using model: gpt-4o-2024-11-20
✅ Saved: /content/drive/My Drive/Colab Notebooks/Run Results/RUN_20250910_191314/outputs/Raw_Response_SINGLE.json
✅ Saved: /content/drive/My Drive/Colab Notebooks/Run Results/RUN_20250910_191314/outputs/Job_Classifications_SINGLE.csv
✅ Saved: /content/drive/My Drive/Colab Notebooks/Run Results/RUN_20250910_191314/outputs/Narrative_SINGLE.txt

=== PAYLOAD SENT (first 800 chars) ===
Objective: Evaluate and group jobs from the "New Sample_08.07.2025.csv" file based on similarities in job functions, not job titles.

Process:

- Compare all jobs against each other using the attributes listed in the file: Education, Work Experience, Licenses/Certifications, Essential Functions, Knowledge, Skills, Abilities, and Position Summary.
- Compare each job with reference sources using the same attributes. I have attached the reference sources for you.
- Group jobs based on similarities into:
  - Major role groupings (e.g., Specialist, Analyst, Manager)
  - Minor sub-g

In [ ]:
# ===== Cell 16.45 — Pre-flight: are all prerequisites loaded for batch? =====
from pathlib import Path

print("Have MODEL_ID:", 'MODEL_ID' in globals(), (MODEL_ID if 'MODEL_ID' in globals() else None))
print("Have df:", 'df' in globals(), (len(df) if 'df' in globals() else None))
print("Have zero_shot_prompt:", 'zero_shot_prompt' in globals())
print("Have OUTPUTS_DIR:", 'OUTPUTS_DIR' in globals(), (OUTPUTS_DIR if 'OUTPUTS_DIR' in globals() else None))

if 'RUN_DIR' in globals():
    print("RUN_DIR:", RUN_DIR)
    print("Outputs path will be:", Path(OUTPUTS_DIR) / "Job_Classifications_Batch.csv")
else:
    print("RUN_DIR missing — re-run your unique run cell (Cell 4).")


Have MODEL_ID: True gpt-4o-2024-11-20
Have df: True 36
Have zero_shot_prompt: True
Have OUTPUTS_DIR: True /content/drive/My Drive/Colab Notebooks/Run Results/RUN_20250910_191314/outputs
RUN_DIR: /content/drive/My Drive/Colab Notebooks/Run Results/RUN_20250910_191314
Outputs path will be: /content/drive/My Drive/Colab Notebooks/Run Results/RUN_20250910_191314/outputs/Job_Classifications_Batch.csv


In [ ]:
from pathlib import Path
(Path(OUTPUTS_DIR)/"Job_Classifications_Batch.csv").unlink(missing_ok=True)
(Path(OUTPUTS_DIR)/"Batch_Errors.json").unlink(missing_ok=True)


# **Changes for Verions 4.8**
>  Supervisor vs Accountant: we use your KSAC lists to separate supervisory signals from accounting signals. Strong accounting + no supervision ⇒ Accountant. Strong supervision with limited scope ⇒ Supervisor (not Manager).
> Director: promotion rules now trigger when the text implies managing managers or executive scope (district/system-wide, strategy/policy, portfolio/budget) Manager & Accountant sub-groups: if blank, we infer I/II/III from duties (supervision + budget/multi-program ⇒ III; supervision only ⇒ II; else I). Overuse of Specialist: if the text looks like Accountant / Analyst / Technician / Coordinator, we nudge away from “Specialist”.
> Overuse of Specialist: if the text looks like Accountant / Analyst / Technician / Coordinator, we nudge away from “Specialist”  



In [ ]:
# ===== Cell 16.5 — Batch v4.8
# Roles constrained to MNPS Roles; KSAC-first; clear Supervisor/Manager/Director rules;
# Supervisor ≠ Accountant guard; Specialist overuse guard;
# Manager/Accountant subgroup fill; minor limited to "", I, II, III, Lead (IV/4 → Lead);
# text-only pipeline, payload per row, resume-safe, fence-clean JSON.

from openai import OpenAI
from pathlib import Path
import pandas as pd, json, time, random, inspect, re, shutil

print("=== Batch v4.8 start ===")

# ---- prerequisites ----
assert 'df' in globals(), "Run Cell 4 first (loads df)."
assert 'INPUTS_DIR' in globals() and 'OUTPUTS_DIR' in globals(), "Run Cell 4 first (sets run folders)."
assert 'MODEL_ID' in globals(), "Run Environment Setup first."
assert 'zero_shot_prompt' in globals(), "Define zero_shot_prompt in your prompt cell."

client = OpenAI(timeout=60.0, max_retries=2)
print("MODEL_ID:", MODEL_ID)
print("Rows in df:", len(df))
print("OUTPUTS_DIR:", OUTPUTS_DIR)

# ---- helpers ----
_FENCE_RE = re.compile(r"^\s*```(?:json)?\s*(.*?)\s*```\s*$", re.I | re.S)
_BRACE_RE = re.compile(r"\{.*\}", re.S)
def coerce_to_json_str(raw: str) -> str:
    if not isinstance(raw, str): return ""
    s = raw.strip()
    m = _FENCE_RE.match(s)
    if m: return m.group(1).strip()
    b = _BRACE_RE.search(s)
    return b.group(0).strip() if b else s

def read_csv_smart_local(p: Path, **kw) -> pd.DataFrame:
    if 'read_csv_smart' in globals():
        return read_csv_smart(p, **kw)
    for enc in ("utf-8", "utf-8-sig", "cp1252", "latin1"):
        try:
            return pd.read_csv(p, encoding=enc, **kw)
        except UnicodeDecodeError:
            continue
    return pd.read_csv(p, encoding="latin1", on_bad_lines="skip", **kw)

# ---- Allowed major roles from MNPS Roles.csv ----
def read_roles_allowed() -> list[str]:
    pin = Path(INPUTS_DIR) / "MNPS Roles.csv"
    pcontent = Path("/content/") / "MNPS Roles.csv"
    path = pin if pin.exists() else (pcontent if pcontent.exists() else None)
    if not path:
        raise FileNotFoundError(f"{pin} and {pcontent} not found. Re-run Cell 4 (copies inputs into this run).")
    df_roles = read_csv_smart_local(path)
    cand = [c for c in df_roles.columns if "major" in c.lower() and "role" in c.lower()] \
        or [c for c in df_roles.columns if "role" in c.lower()]
    col = cand[0]
    roles = sorted({str(x).strip() for x in df_roles[col].dropna().unique() if str(x).strip()})
    return roles

ALLOWED_MAJORS = read_roles_allowed()
ALLOWED_MAJORS_SET = set(ALLOWED_MAJORS)
print("Allowed majors:", ALLOWED_MAJORS)

# ---- Optional tiny Ground Truth context ----
context_block = ""
gt = Path(INPUTS_DIR) / "Ground Truth Masterfile.csv"
if gt.exists():
    try:
        gdf = read_csv_smart_local(gt).fillna("")
        pref = [
            "Original Job Title","New Job Title","Major Role Group","Minor Sub-Group","Justification for Grouping",
            "Position Summary","Education","Work Experience","Licenses and Certifications","Essential Functions","Knowledge, Skills and Abilities"
        ]
        cols = [c for c in pref if c in gdf.columns] or list(gdf.columns)[:8]
        context_block = "Context (3 ground-truth examples):\n" + gdf[cols].head(3).to_json(orient="records", force_ascii=False)
        print("Context chars:", len(context_block))
    except Exception as e:
        context_block = f"(Context unavailable: {e})"
else:
    print("No Ground Truth Masterfile found (optional).")

# ---- Title anonymization for PS/EF only ----
def anonymize(title: str, text: str) -> str:
    if not title: return text
    toks = [t for t in title.strip().split() if t]
    if not toks: return text
    pattern = r"(?i)\b" + r"(?:[\s\-]+)".join(map(re.escape, toks)) + r"(?:'s)?\b"
    return re.sub(pattern, "this role", text)

# ---- Signals tuned to your KSACs + new hierarchy guidance ----
SUP_KSAC_WORDS = [
    "team management","performance evaluation","coaching","training and development",
    "staff development","conflict resolution","assigns work","leads a team","leadership",
    "supervise","supervises","supervision","discipline","mentors","onboarding","appraisals",
    "performance reviews","scheduling","timesheets","work assignments","interpersonal skills",
    "policy enforcement","procedures compliance","frontline","quality"
]
ACCT_WORDS = [
    "gaap","general ledger","g/l","journal entries","reconciliation","reconcile","reconciling",
    "month-end close","monthly close","close","audit","auditor","financial reporting",
    "financial statements","accounts payable","accounts receivable","payables","receivables",
    "ap","a/p","ar","a/r","gl","fund accounting","grant accounting","ledger","variance analysis",
    "balance sheet","income statement","cash flow"
]
ANALYST_HINTS = ["analyze","analysis","analytics","kpi","dashboard","reporting","research","sql","insight","metrics","statistical"]
COORD_HINTS   = ["coordinate","coordination","logistics","arrangements","liaison","calendar","events","scheduling"]
TECH_HINTS    = ["install","repair","troubleshoot","maintenance","equipment","hardware","software","configuration"]

# Manager-level planning/budget/staffing/outcomes
MGR_PLANNING = ["plan","planning","workforce planning","staffing","resource allocation","program management","operational plan","roadmap","outcomes","targets","kpi","performance targets","budget","budgeting"]

# Director-level strategy/policy/portfolio signals (in addition to regex below)
DIR_STRATEGIC = ["department","division","strategic plan","strategy","vision","policy","governance","portfolio","district-wide","system-wide","enterprise","multi-year","sets direction","sets policy"]

SIG_RE = {
    "manages_managers": re.compile(r"\b(manag\w+\s+(other\s+)?managers|supervis\w+\s+(other\s+)?supervisors)\b", re.I),
    "supervises_staff": re.compile(r"\b(supervis\w+|direct reports|leads a team|performance reviews|assigns work|coaching|hiring|hire|disciplin\w+)\b", re.I),
    "district_wide":    re.compile(r"\b(district[\-\s]?wide|system[\-\s]?wide|across departments|cross[-\s]?functional)\b", re.I),
    "strategy_policy":  re.compile(r"\b(strateg(y|ic)|policy|governance|portfolio)\b", re.I),
    "budget_owner":     re.compile(r"\b(budget(s)?|fiscal management|approv\w+\s+expenditures|budget authority|financial oversight|grants? administration)\b", re.I),
    "multi_program":    re.compile(r"\b(multiple programs|program portfolio|across programs|enterprise)\b", re.I),
    "exec_words":       re.compile(r"\b(executive|cabinet|reports to (chief|superintendent|assistant superintendent|director))\b", re.I),
    "coord_keywords":   re.compile(r"\b(coordinat\w+)\b", re.I),
    "students":         re.compile(r"\bstudents?\b", re.I),
}

def count_hits(words, text_lower):
    return sum(1 for w in words if re.search(rf"\b{re.escape(w)}\b", text_lower))

def detect_signals(ps: str, ef: str, ksa: str) -> dict:
    t = " ".join([(ps or ""), (ef or ""), (ksa or "")])
    low = t.lower()
    sigs = {k: bool(r.search(t)) for k, r in SIG_RE.items()}
    sigs["sup_ksac_count"] = count_hits(SUP_KSAC_WORDS, low)
    sigs["acct_count"]     = count_hits(ACCT_WORDS, low)
    sigs["analyst_count"]  = count_hits(ANALYST_HINTS, low)
    sigs["coord_count"]    = count_hits(COORD_HINTS, low)
    sigs["tech_count"]     = count_hits(TECH_HINTS, low)
    sigs["mgr_plan_count"] = count_hits(MGR_PLANNING, low)
    sigs["dir_strat_count"]= count_hits(DIR_STRATEGIC, low)
    return sigs

# ---- Build per-row text (ignore Job Description Name in evaluation; anonymize PS/EF) ----
def build_row_text(r):
    def getv(c):
        try: v = r[c]; return "" if pd.isna(v) else str(v)
        except Exception: return ""
    title = getv("Job Description Name")  # For output only; not evaluated
    ps = anonymize(title, getv("Position Summary"))
    ef = anonymize(title, getv("Essential Functions"))
    ksa = getv("Knowledge, Skills and Abilities")
    body = (
        f"Position Summary: {ps}\n"
        f"Education: {getv('Education')}\n"
        f"Work Experience: {getv('Work Experience')}\n"
        f"Licenses and Certifications: {getv('Licenses and Certifications')}\n"
        f"Essential Functions: {ef}\n"
        f"Knowledge, Skills and Abilities: {ksa}\n"
    )
    s = detect_signals(ps, ef, ksa)
    students_hint = s["students"]
    constraints = (
        "Hard constraints:\n"
        f"- Major Role Group MUST be one of: {', '.join(ALLOWED_MAJORS)}\n"
        "- Use MNPS Roles and MNPS KSACs as the primary basis for Major Role Group determination.\n"
        "- Use Competency Extended Descriptions and Korn/Ferry 38 Competencies only to clarify/support the justification.\n"
        "- Use Ground Truth Masterfile as a tie-breaker when multiple roles are plausible by KSACs.\n"
        "- Minor Sub-Group must be one of: \"\", I, II, III, Lead (no higher than Lead). Use Lead only when the record clearly indicates a level IV scope.\n"
        "- Teacher vs Instructor: if KSAC alignment is equal and the text references students, prefer Teacher.\n"
        "- Hierarchy definitions:\n"
        "    • Supervisor: first-line people leader overseeing day-to-day operations, scheduling, quality, and policy enforcement for a unit; may recommend but often does not own hiring/firing; tactical focus.\n"
        "    • Manager: mid-level leader responsible for planning, budgeting, staffing, and outcomes for a function or multiple teams; often supervises supervisors and translates strategy into operations.\n"
        "    • Director: senior leader accountable for a department/division, aligning area goals with district strategy, stewarding significant budgets/resources, and setting policy and long-term direction.\n"
        "- Advisory family: choose among Advisor / Consultant / Coach / Facilitator based on KSAC alignment and determinant factors; justify with KSAC evidence.\n"
        "- Specialist is a narrow technical IC role—use only when KSACs clearly indicate deep specialization and no other role (Analyst, Accountant, Advisor, Technician, Coordinator) fits better.\n"
    )
    if students_hint:
        constraints += "Note: The text references students; prefer Teacher vs Instructor when KSACs are equivalent.\n"
    full = zero_shot_prompt.strip() + "\n\n" + constraints
    if context_block: full += "\n" + context_block
    full += "\n\nClassify the following job description (title ignored for evaluation):\n\n" + body
    return title, full, s, ps, ef, ksa

# ---- SDK capability detection ----
def _has_param(obj, name):
    try: return name in inspect.signature(obj).parameters
    except Exception: return False
supports_parse_schema  = _has_param(client.responses.parse,  "response_format")
supports_create_schema = _has_param(client.responses.create, "response_format")
print("supports_parse_schema:", supports_parse_schema, "| supports_create_schema:", supports_create_schema)

def call_model(text, temp, max_tokens):
    if supports_parse_schema:
        resp = client.responses.parse(
            model=MODEL_ID,
            input=[{"role":"user","content":[{"type":"input_text","text":text}]}],
            temperature=temp, max_output_tokens=max_tokens,
            response_format=JobClassificationTable,
        )
        return resp.output_parsed, (resp.output_text or "")
    elif supports_create_schema:
        schema = JobClassificationTable.model_json_schema()
        resp = client.responses.create(
            model=MODEL_ID,
            input=[{"role":"user","content":[{"type":"input_text","text":text}]}],
            temperature=temp, max_output_tokens=max_tokens,
            response_format={"type":"json_schema","json_schema":{"name":"JobClassificationTable","schema":schema,"strict":True}},
        )
        raw = coerce_to_json_str(getattr(resp,"output_text",None) or "")
        return JobClassificationTable.model_validate_json(raw), raw
    else:
        schema_json = json.dumps(JobClassificationTable.model_json_schema(), indent=2)
        strict = (
            "You MUST return ONLY valid JSON that matches the JSON Schema. No prose, no markdown.\n"
            f"JSON Schema:\n{schema_json}\n\nTask:\n{text}"
        )
        resp = client.responses.create(
            model=MODEL_ID,
            input=[{"role":"user","content":[{"type":"input_text","text":strict}]}],
            temperature=temp, max_output_tokens=max_tokens,
        )
        raw = coerce_to_json_str(getattr(resp,"output_text",None) or "")
        return JobClassificationTable.model_validate_json(raw), raw

# ---- post-normalization ----
MINOR_ALLOWED = {"", "I", "II", "III", "LEAD"}

def norm_minor(s: str) -> str:
    """Normalize minor sub-group; map IV/4/Level 4 to Lead; cap at Lead."""
    if s is None: return ""
    x = str(s).strip().upper().replace("LEVEL ", "")
    map_ = {
        "": "", "0":"",
        "1":"I","I":"I",
        "2":"II","II":"II",
        "3":"III","III":"III",
        "4":"LEAD","IV":"LEAD","LEAD":"LEAD","L4":"LEAD"
    }
    v = map_.get(x, x)
    return v if v in MINOR_ALLOWED else ""

def choose_hierarchy_label(s: dict) -> str | None:
    """Return best-fit among Director/Manager/Supervisor given signals; None if inconclusive."""
    # Weighted scores
    dir_score = (
        (2 if s.get("manages_managers") else 0) +
        (1 if s.get("exec_words") else 0) +
        (1 if s.get("district_wide") else 0) +
        (1 if s.get("strategy_policy") else 0) +
        (1 if s.get("budget_owner") else 0) +
        (1 if s.get("multi_program") else 0) +
        (1 if s.get("dir_strat_count",0) >= 1 else 0)
    )
    mgr_score = (
        (1 if s.get("supervises_staff") else 0) +
        (1 if s.get("budget_owner") else 0) +
        (1 if s.get("multi_program") else 0) +
        (1 if s.get("mgr_plan_count",0) >= 1 else 0)
    )
    sup_score = (
        (1 if s.get("supervises_staff") else 0) +
        (1 if s.get("sup_ksac_count",0) >= 2 else 0) +
        (1 if not s.get("budget_owner") else 0) +
        (1 if not (s.get("multi_program") or s.get("strategy_policy") or s.get("district_wide")) else 0)
    )

    # Hard promotion if manages managers
    if s.get("manages_managers") and "Director" in ALLOWED_MAJORS_SET:
        return "Director"

    # Pick max score with simple precedence Director > Manager > Supervisor
    choices = []
    if "Director" in ALLOWED_MAJORS_SET:  choices.append(("Director", dir_score))
    if "Manager" in ALLOWED_MAJORS_SET:   choices.append(("Manager",  mgr_score))
    if "Supervisor" in ALLOWED_MAJORS_SET:choices.append(("Supervisor",sup_score))
    if not choices:
        return None
    choices.sort(key=lambda x: x[1], reverse=True)
    # if tie or all zeros, return None to avoid forcing
    top_label, top_score = choices[0]
    if top_score == 0 or (len(choices)>1 and top_score == choices[1][1]):
        return None
    return top_label

def upgrade_major_with_signals(current: str, s: dict) -> str:
    """Apply hierarchy decision + other family guards (Teacher/Instructor, Accountant vs Supervisor, Specialist drift)."""
    if not current:
        current = ""
    cur = current.strip()
    low = cur.lower()

    # Teacher vs Instructor preference
    if low in {"teacher","instructor"} and s.get("students") and ("Teacher" in ALLOWED_MAJORS_SET):
        return "Teacher"

    # Clear accounting without supervision -> Accountant
    if s.get("acct_count", 0) >= 2 and not s.get("supervises_staff") and ("Accountant" in ALLOWED_MAJORS_SET):
        return "Accountant"

    # Hierarchy decision
    best_hierarchy = choose_hierarchy_label(s)
    if best_hierarchy and best_hierarchy in ALLOWED_MAJORS_SET:
        return best_hierarchy

    # Coordinator with no supervision
    if s.get("coord_keywords") and not s.get("supervises_staff") and ("Coordinator" in ALLOWED_MAJORS_SET):
        return "Coordinator"

    # Specialist overuse guard → push to Accountant/Analyst/Technician/Coordinator when signals strong
    if low == "specialist":
        if s.get("acct_count",0) >= 2 and ("Accountant" in ALLOWED_MAJORS_SET):   return "Accountant"
        if s.get("analyst_count",0) >= 2 and ("Analyst" in ALLOWED_MAJORS_SET):   return "Analyst"
        if s.get("tech_count",0)    >= 2 and ("Technician" in ALLOWED_MAJORS_SET):return "Technician"
        if s.get("coord_count",0)   >= 2 and ("Coordinator" in ALLOWED_MAJORS_SET):return "Coordinator"

    return cur or current

def norm_major(current: str, s: dict) -> str:
    """Snap to allowed list; apply upgrade logic above."""
    t = upgrade_major_with_signals(current, s)
    if t in ALLOWED_MAJORS_SET: return t
    low = t.lower()
    for a in ALLOWED_MAJORS:
        if a.lower() == low:
            return a
    tok = set(re.findall(r"\w+", low))
    best, score = None, -1
    for a in ALLOWED_MAJORS:
        sc = len(tok & set(re.findall(r"\w+", a.lower())))
        if sc > score:
            best, score = a, sc
    return best or t

def fill_minor_if_blank(major: str, minor: str, s: dict, ps: str, ef: str) -> str:
    """If minor is blank for Manager/Accountant, infer I/II/III from duties/signals."""
    if (minor or "").strip():
        return norm_minor(minor)
    if major == "Manager":
        sup = s.get("supervises_staff", False)
        budget = s.get("budget_owner", False)
        multi = s.get("multi_program", False) or s.get("district_wide", False) or s.get("strategy_policy", False)
        if sup and (budget or multi): return "III"
        if sup: return "II"
        return "I"
    if major == "Accountant":
        txt_low = ((ps or "") + " " + (ef or "")).lower()
        if re.search(r"\b(policy|complex consolidations?|external audit|audit lead|enterprise|governance)\b", txt_low): return "III"
        if re.search(r"\b(reconcil\w+|journal entries|monthly close|closing|grant accounting|financial reports?|payables|receivables|gl|general ledger)\b", txt_low): return "II"
        return "I"
    return norm_minor(minor)

# ---- parameters ----
ROW_START   = 0
ROW_LIMIT   = None      # None → process all
TEMP        = 0.2
MAX_TOKENS  = 900
SAVE_EVERY  = 20
MAX_ATTEMPTS_PER_ROW = 3

batch_csv_path = Path(OUTPUTS_DIR) / "Job_Classifications_Batch.csv"
processed = set()
if batch_csv_path.exists():
    try:
        prior = pd.read_csv(batch_csv_path, usecols=["source_row_index"])
        processed = set(prior["source_row_index"].astype(int).tolist())
        print(f"Resume mode: {len(processed)} rows already done; will skip them.")
    except Exception as e:
        print("Resume disabled:", e)

PAYLOADS_DIR = Path(OUTPUTS_DIR) / "api_payloads"
PAYLOADS_DIR.mkdir(parents=True, exist_ok=True)
payload_index = Path(OUTPUTS_DIR) / "API_Payloads.ndjson"

end_idx = len(df) if ROW_LIMIT is None else min(len(df), ROW_START + ROW_LIMIT)
indexes = [i for i in range(ROW_START, end_idx) if i not in processed]
print(f"Planned rows to process: {len(indexes)} of {len(df)} (from {ROW_START} to {end_idx-1})")

records, errors = [], []
start = time.time()

for k, i in enumerate(indexes, start=1):
    r = df.iloc[i]
    # Build text (ignore title in eval; anonymize PS/EF)
    def getv(c):
        try: v = r[c]; return "" if pd.isna(v) else str(v)
        except Exception: return ""
    original_title = getv("Job Description Name")
    title, text, signals, ps, ef, ksa = build_row_text(r)

    # Save the exact payload used
    pp = PAYLOADS_DIR / f"row_{i:04d}.txt"
    pp.write_text(text, encoding="utf-8")
    with open(payload_index, "a", encoding="utf-8") as f:
        f.write(json.dumps({"row_index": i, "payload_file": str(pp), "chars": len(text)}) + "\n")

    parsed, raw = None, ""
    for attempt in range(MAX_ATTEMPTS_PER_ROW):
        try:
            parsed, raw = call_model(text, TEMP, MAX_TOKENS)
            break
        except Exception as e:
            if attempt == MAX_ATTEMPTS_PER_ROW - 1:
                errors.append((i, "exception", str(e)[:500]))
            time.sleep(min(20, 1.8**attempt + random.random()))

    if parsed:
        try:
            for rec in parsed.job_classification_table:
                row = rec.model_dump()
                row["job_title_original"] = original_title or row.get("job_title_original","")
                row["major_role_group"]   = norm_major(row.get("major_role_group",""), signals)
                row["minor_sub_group"]    = fill_minor_if_blank(row["major_role_group"], row.get("minor_sub_group",""), signals, ps, ef)
                row["source_row_index"]   = i
                row["model_used"]         = MODEL_ID
                records.append(row)
        except Exception as e:
            errors.append((i, "collect_error", str(e)[:300]))
    else:
        cleaned = coerce_to_json_str(raw) if raw else ""
        errors.append((i, "no_parsed_output", cleaned[:300]))

    # checkpoint save
    if (k % SAVE_EVERY == 0) or (k == len(indexes)):
        if records:
            if batch_csv_path.exists():
                prev = pd.read_csv(batch_csv_path)
                merged = pd.concat([prev, pd.DataFrame(records)], ignore_index=True)
                merged.drop_duplicates(subset=["source_row_index","job_title_original","new_job_title"], inplace=True)
                merged.to_csv(batch_csv_path, index=False, encoding="utf-8")
            else:
                pd.DataFrame(records).to_csv(batch_csv_path, index=False, encoding="utf-8")
            print(f"Checkpoint: wrote {len(pd.read_csv(batch_csv_path))} rows to batch CSV.")
            records = []
        Path(OUTPUTS_DIR, "Batch_Errors.json").write_text(json.dumps(errors, indent=2), encoding="utf-8")

    print(f"[{k}/{len(indexes)}] row {i} | ok {k-len(errors)} | err {len(errors)}")

# mirror to single name for housekeeping
if batch_csv_path.exists():
    shutil.copy2(batch_csv_path, Path(OUTPUTS_DIR) / "Job_Classifications.csv")
    print("📄 Copied batch to:", Path(OUTPUTS_DIR) / "Job_Classifications.csv")

print("✅ Batch complete. Files in:", OUTPUTS_DIR)
print("Payloads dir:", PAYLOADS_DIR)


# **Changes for Verions 4.2**
> Batch audit: counts, parameters, error preview   
> Sanity Check  

In [ ]:
# ===== Cell 16.55 — Batch audit: counts, parameters, error preview =====
from pathlib import Path
import pandas as pd, json

assert 'OUTPUTS_DIR' in globals(), "Run Cell 4 first (creates OUTPUTS_DIR)."
assert 'df' in globals(), "Run Cell 4 first (loads df)."

print("Total rows in input df:", len(df))

batch_csv_path = Path(OUTPUTS_DIR) / "Job_Classifications_Batch.csv"
if batch_csv_path.exists():
    dfb = pd.read_csv(batch_csv_path)
    print("Rows saved in batch CSV:", len(dfb))
    if "source_row_index" in dfb.columns:
        done = sorted(dfb["source_row_index"].astype(int).unique().tolist())
        print("First 10 processed row indexes:", done[:10])
        print("Last 10 processed row indexes:", done[-10:])
    else:
        print("Note: 'source_row_index' column missing in batch CSV.")
else:
    print("⚠️ No batch CSV found at:", batch_csv_path)

errors_path = Path(OUTPUTS_DIR) / "Batch_Errors.json"
if errors_path.exists():
    try:
        errs = json.loads(errors_path.read_text())
        print("Error entries:", len(errs))
        for j, e in enumerate(errs[:5]):
            print(f"  {j+1}.", e if isinstance(e, str) else (e[0:2] if isinstance(e, list) else e))
    except Exception as e:
        print("Could not read Batch_Errors.json:", e)
else:
    print("No Batch_Errors.json present — either none failed or nothing ran.")


Total rows in input df: 36
Rows saved in batch CSV: 36
First 10 processed row indexes: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
Last 10 processed row indexes: [26, 27, 28, 29, 30, 31, 32, 33, 34, 35]
Error entries: 0


In [ ]:
# ===== Cell 16.6 — Quick sanity check for current run =====
from pathlib import Path
import pandas as pd, json

assert 'OUTPUTS_DIR' in globals(), "Run your unique-run cell first (defines OUTPUTS_DIR)."

batch = Path(OUTPUTS_DIR) / "Job_Classifications_Batch.csv"
if batch.exists():
    dfb = pd.read_csv(batch)
    print("✅ Batch rows in this run:", len(dfb))
    display(dfb.head(5))
else:
    print("⚠️ No batch file found at", batch)

errs = Path(OUTPUTS_DIR) / "Batch_Errors.json"
if errs.exists():
    e = json.loads(Path(errs).read_text())
    print("⚠️ Rows with errors:", len(e))
    if e:
        print("First error:", e[0])


✅ Batch rows in this run: 36


,job_title_original,new_job_title,major_role_group,minor_sub_group,grouping_justification,source_row_index,model_used
0,Tech Mail Center,Mail Center Specialist I,Specialist,I,The role involves specialized functions such a...,0,gpt-4o-2024-11-20
1,Coord Safe and Drug Free,Student Support Program Coordinator I,Coordinator,I,The role involves coordinating district-wide p...,1,gpt-4o-2024-11-20
2,Application Systems Analyst- Kronos,Kronos Application Specialist II,Specialist,II,The role requires subject matter expertise in ...,2,gpt-4o-2024-11-20
3,Assistant Financial Analyst,Financial Analyst I,Analyst,I,The role involves assisting in financial analy...,3,gpt-4o-2024-11-20
4,Analyst Payroll Compliance,Payroll Audit Specialist II,Specialist,II,"The role focuses on auditing payroll data, tra...",4,gpt-4o-2024-11-20


⚠️ Rows with errors: 0


In [ ]:
#Cell 17
# Inspect parsed output (Responses API)
try:
    parsed  # from Cell 16
    print(parsed.model_dump_json(indent=2))
except NameError:
    print("No 'parsed' object found. Run Cell 16 first.")


We can make this into a table using pandas!

In [ ]:
# Cell 17.5 — Build a response_dict from the Responses API parsed object
from pathlib import Path
import json
import pandas as pd

# Make sure Cell 16 ran (it defines `parsed`) and the run folders exist
assert 'parsed' in globals(), "Run Cell 16 first (it sets `parsed`)."
assert 'OUTPUTS_DIR' in globals(), "Run the unique-run cell first (defines OUTPUTS_DIR)."

# Convert the Pydantic objects to plain dicts
response_dict = {
    "job_classification_table": [rec.model_dump() for rec in parsed.job_classification_table],
    "narrative_rationale": parsed.narrative_rationale,
}

# Optional: preview the first rows
display(pd.DataFrame(response_dict["job_classification_table"]).head(10))

# Optional: save a pretty JSON alongside your other outputs
out_json = Path(OUTPUTS_DIR) / "Parsed_Response.json"
out_json.write_text(json.dumps(response_dict, indent=2), encoding="utf-8")
print("Saved:", out_json)

# Also return the dict so it shows below the cell
response_dict


In [ ]:
#Cell 18
# Preview the saved classifications CSV (if present)
from pathlib import Path
import pandas as pd

csv_path = Path(OUTPUTS_DIR) / "Job_Classifications.csv"
if csv_path.exists():
    display(pd.read_csv(csv_path).head(10))
else:
    print("No Job_Classifications.csv found in", OUTPUTS_DIR)


In [ ]:
# Cell 19 ===== Housekeeping & Archive (Run Results) =====
# Place this cell at the END of the notebook. Run after your pipeline finishes.
from google.colab import drive
from pathlib import Path
import shutil, json, re
import datetime as dt
import pandas as pd

# ---------- CONFIG (edit to taste) ----------
RUN_ROOT = Path("/content/drive/My Drive/Colab Notebooks/Run Results")
ARCHIVE_DIR = RUN_ROOT / "_archives"
MASTER_DIR  = RUN_ROOT / "_master"

KEEP_LAST_N_RUNS   = 10     # keep this many newest runs; older ones can be deleted
ZIP_OLDER_RUNS     = True   # zip runs (into _archives) to save space
PURGE_RAW_JSON     = True   # delete outputs/Raw_Response.json inside each run
PURGE_PARSED_JSON  = False  # delete outputs/Parsed_Response.json
PURGE_BATCH_ERRORS = False  # delete outputs/Batch_Errors.json
SKIP_CURRENT_RUN   = True   # don't zip/purge/delete the most recent run
DRY_RUN            = True   # <<< safety: set False to actually apply changes

# ---------- Mount Drive (no-op if already mounted) ----------
drive.mount('/content/drive')

# ---------- Helpers ----------
def parse_run_ts(name: str):
    m = re.match(r"RUN_(\d{8}_\d{6})$", name)
    if not m:
        return None
    try:
        return dt.datetime.strptime(m.group(1), "%Y%m%d_%H%M%S")
    except Exception:
        return None

def folder_size_bytes(p: Path) -> int:
    total = 0
    for f in p.rglob("*"):
        if f.is_file():
            try:
                total += f.stat().st_size
            except Exception:
                pass
    return total

def human_mb(nbytes: int) -> str:
    return f"{nbytes/1_000_000:.2f} MB"

# ---------- Discover run folders ----------
runs = []
for d in RUN_ROOT.iterdir():
    if d.is_dir() and d.name.startswith("RUN_"):
        ts = parse_run_ts(d.name)
        if ts:
            runs.append((d, ts))

runs.sort(key=lambda x: x[1], reverse=True)  # newest first
print(f"Found {len(runs)} run folders under: {RUN_ROOT}")

current = runs[0][0] if runs else None
if current:
    print("Most recent run:", current.name)

# Summary of the first few
for d, ts in runs[:5]:
    print(f" - {d.name} | {ts:%Y-%m-%d %H:%M:%S} | size≈ {human_mb(folder_size_bytes(d))}")

# Ensure archive/master dirs
ARCHIVE_DIR.mkdir(parents=True, exist_ok=True)
MASTER_DIR.mkdir(parents=True, exist_ok=True)

# ---------- Plan actions ----------
actions = []

# 1) Purge large intermediates within runs
def plan_purges(d: Path):
    out = d / "outputs"
    if not out.exists():
        return
    if PURGE_RAW_JSON and (out / "Raw_Response.json").exists():
        actions.append(("delete_file", out / "Raw_Response.json"))
    if PURGE_PARSED_JSON and (out / "Parsed_Response.json").exists():
        actions.append(("delete_file", out / "Parsed_Response.json"))
    if PURGE_BATCH_ERRORS and (out / "Batch_Errors.json").exists():
        actions.append(("delete_file", out / "Batch_Errors.json"))

# 2) Zip older runs (into _archives)
def plan_zip(d: Path):
    z = ARCHIVE_DIR / f"{d.name}.zip"
    if not z.exists():
        actions.append(("zip_folder", (d, z)))

# 3) Delete runs beyond retention
to_prune = runs[KEEP_LAST_N_RUNS:] if KEEP_LAST_N_RUNS is not None else []
for d, ts in runs:
    if SKIP_CURRENT_RUN and current and d == current:
        continue
    # Purges
    plan_purges(d)
    # Zip plan
    if ZIP_OLDER_RUNS:
        plan_zip(d)

for d, ts in to_prune:
    actions.append(("delete_folder", d))

# ---------- Show plan ----------
print("\nPlanned actions:")
if not actions:
    print(" (none)")
else:
    for act, obj in actions:
        if act == "zip_folder":
            d, z = obj
            print(f" - ZIP {d.name}  →  {z.name}")
        else:
            print(f" - {act.upper()}: {obj}")

# ---------- Execute (unless DRY_RUN) ----------
if DRY_RUN:
    print("\nDRY_RUN=True — no changes applied. Set DRY_RUN=False to execute.")
else:
    for act, obj in actions:
        try:
            if act == "delete_file":
                Path(obj).unlink(missing_ok=True)
            elif act == "zip_folder":
                d, z = obj
                # create zip in ARCHIVE_DIR; shutil.make_archive adds .zip automatically
                base_name = z.with_suffix("")  # remove .zip for make_archive
                shutil.make_archive(str(base_name), 'zip', root_dir=d)
            elif act == "delete_folder":
                shutil.rmtree(obj, ignore_errors=True)
        except Exception as e:
            print("  ! Error:", act, obj, e)
    print("\n✅ Housekeeping complete.")

# ---------- Aggregate a master CSV across all runs (safe to do anytime) ----------
frames = []
for d, ts in runs:
    for name in ["Job_Classifications_Batch.csv", "Job_Classifications.csv"]:
        csvp = d / "outputs" / name
        meta = d / "RUN_METADATA.json"
        if csvp.exists():
            try:
                df_run = pd.read_csv(csvp)
                df_run["run_folder"]  = d.name
                df_run["source_file"] = name
                # enrich with metadata if available
                if meta.exists():
                    try:
                        m = json.loads(meta.read_text())
                        df_run["created_utc"] = m.get("created_utc")
                        df_run["model_used"]  = m.get("resolved_model_id") or m.get("model_used")
                    except Exception:
                        pass
                frames.append(df_run)
            except Exception as e:
                print(f"  ! Skipping {csvp.name} due to read error:", e)

if frames:
    master = pd.concat(frames, ignore_index=True)
    MASTER_DIR.mkdir(parents=True, exist_ok=True)
    master_out = MASTER_DIR / "All_Job_Classifications.csv"
    master.to_csv(master_out, index=False, encoding="utf-8")
    print(f"\n📚 Master CSV updated: {master_out} ({len(master)} rows; from {len(frames)} files)")
else:
    print("\n(No job classification CSVs found to aggregate.)")
